# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is defined by a Croissant schema available via the following URL.

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using the `mlcroissant` library.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL for dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# The .metadata object exposes .name and .description attributes and more.
print(f"Dataset name: {dataset.metadata.name}\nDescription: {dataset.metadata.description}")

## 2. Data Overview
List available record sets, fields, and their unique `@id` references. For further analysis, use `@id` attributes for each data entity.


In [ ]:
# List all available record sets and their fields by @id
if hasattr(dataset, "record_sets"):
    record_sets = dataset.record_sets
    print(f"Number of record sets: {len(record_sets)}")
    for rset in record_sets:
        print(f"RecordSet name: {getattr(rset, 'name', None)} | @id: {rset.id}")
        print("  Fields:")
        for field in getattr(rset, 'fields', []):
            print(f"    Field name: {getattr(field, 'name', None)} | @id: {field.id}")
else:
    print("No record sets detected in this dataset")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Record set and field `@id`s are referenced from the section above.


In [ ]:
# Example: extract all record sets into DataFrames
dataframes = {}

# Build a mapping of record set @id to their names
if hasattr(dataset, "record_sets"):
    record_sets = dataset.record_sets
    record_set_ids = [rset.id for rset in record_sets]
    record_set_names = {rset.id: rset.name for rset in record_sets}

    for rset in record_sets:
        records = list(dataset.records(record_set=rset.id))
        if records:
            df = pd.DataFrame(records)
        else:
            df = pd.DataFrame()
        dataframes[rset.id] = df
        print(f"Loaded RecordSet {rset.name} (@id: {rset.id}). Shape: {df.shape}")

    # Display the first DataFrame and its columns if any record set was loaded
    if record_set_ids:
        main_record_set_id = record_set_ids[0]
        print(f"\nColumns for record set '{record_set_names[main_record_set_id]}' (@id: {main_record_set_id}):")
        print(dataframes[main_record_set_id].columns.tolist())
        dataframes[main_record_set_id].head()
else:
    print("No record sets available for extraction.")

## 4. Exploratory Data Analysis (EDA)
Apply exploratory analysis such as filtering, normalization, and grouping using specific fields referenced by `@id`.


In [ ]:
# Example EDA for the first record set (if available and with numeric fields)
import numpy as np

if 'main_record_set_id' in locals():
    df = dataframes[main_record_set_id]
    # Identify numeric fields (columns) by type (float/int)
    numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    print(f"Numeric fields in record set {main_record_set_id}: {numeric_fields}")

    if numeric_fields:
        numeric_field_id = numeric_fields[0]  # Use the first numeric field for demonstration
        threshold = df[numeric_field_id].mean()  # Use mean as automatic threshold
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df[[numeric_field_id]].head())

        # Normalize column
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized '{numeric_field_id}' for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].copy()].head())

        # Try grouping by non-numeric field if available
        other_fields = [col for col in df.columns if col != numeric_field_id]
        group_field = None
        for col in other_fields:
            if df[col].dtype == 'object':
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"\nGrouped filtered data by '{group_field}':")
            print(grouped_df.head())
        else:
            print("No categorical field found for grouping.")
    else:
        print("No numeric fields available for EDA.")
else:
    print("No main record set DataFrame available for EDA.")

## 5. Visualization
Visualize the distribution of the first available numeric field, or relationships between fields, using matplotlib.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'main_record_set_id' in locals() and numeric_fields:
    plt.figure(figsize=(8,4))
    sns.histplot(dataframes[main_record_set_id][numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id} in Record Set '@id={main_record_set_id}'")
    plt.xlabel(numeric_field_id)
    plt.tight_layout()
    plt.show()
    
    # If a categorical group_field exists, plot boxplot
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=dataframes[main_record_set_id][group_field], y=dataframes[main_record_set_id][numeric_field_id])
        plt.title(f"Boxplot of {numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("No suitable numeric field available for visualization.")

## 6. Conclusion
We demonstrated loading, schema inspection, and basic analysis on the FAIR² dataset using `mlcroissant`.

- Data was referenced strictly via `@id` fields to ensure reproducibility and schema consistency.
- We performed EDA with filtering and normalization on available numeric fields.
- Basic visualizations showed feature distributions and relationships.

This notebook can be extended with more domain-specific statistics or machine learning workflows as needed.